# PySpark DataFrame Data Engineering Project — Media Analytics

Business-oriented PySpark DataFrame project using two related input files.

**Input files:** `content.csv`, `views.csv`

**Questions:** 30

Topics include DataFrame creation, schema, select, filter, withColumn, when, cast, groupBy, agg, joins, orderBy, window functions, null handling, duplicates, date functions, pivot, union-style reporting, and business KPIs.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("PySparkDataFrameProject").getOrCreate()

df1 = spark.read.option("header", True).option("inferSchema", True).csv("content.csv")
df2 = spark.read.option("header", True).option("inferSchema", True).csv("views.csv")

df1.show(5, truncate=False)
df2.show(5, truncate=False)


## Q1. Inspect Schemas

Display both DataFrame schemas and row counts.

In [ ]:
df1.printSchema()
df2.printSchema()
print(df1.count(), df2.count())

## Q2. Select Business Columns

Select content_id and content_type from the first DataFrame.

In [ ]:
df1.select('content_id','content_type').show(20, truncate=False)

## Q3. Filter Records

Filter the second DataFrame where ad_revenue is greater than its average.

In [ ]:
avg_value = df2.select(avg('ad_revenue').alias('avg')).first()['avg']
df2.filter(col('ad_revenue') > avg_value).show(20, truncate=False)

## Q4. Create Derived Column

Create a new column based on ad_revenue using a High/Normal classification.

In [ ]:
df2.withColumn('value_category', when(col('ad_revenue') >= avg('ad_revenue').over(Window.partitionBy()), 'High').otherwise('Normal')).show(20, truncate=False)

## Q5. Null Analysis

Show null counts for every column in both DataFrames.

In [ ]:
df1.select([sum(col(c).isNull().cast('int')).alias(c) for c in df1.columns]).show()
df2.select([sum(col(c).isNull().cast('int')).alias(c) for c in df2.columns]).show()

## Q6. Handle Null Values

Fill null values in content_type and viewer_city with 'Unknown'.

In [ ]:
df1.fillna({'content_type': 'Unknown'}).show(5)
df2.fillna({'viewer_city': 'Unknown'}).show(5)

## Q7. Remove Duplicates

Remove duplicate records and compare row counts.

In [ ]:
print('Before:', df1.count(), df2.count())
print('After:', df1.dropDuplicates().count(), df2.dropDuplicates().count())

## Q8. Inner Join

Join the two DataFrames using content_id and content_id.

In [ ]:
joined = df1.join(df2, df1['content_id'] == df2['content_id'], 'inner')
joined.show(20, truncate=False)

## Q9. Left Join

Perform a left join and identify unmatched records.

In [ ]:
left = df1.join(df2, df1['content_id'] == df2['content_id'], 'left')
left.show(20, truncate=False)

## Q10. Group By Category

Calculate count and average ad_revenue by viewer_city.

In [ ]:
df2.groupBy('viewer_city').agg(count('*').alias('records'), avg('ad_revenue').alias('avg_value')).orderBy(desc('avg_value')).show()

## Q11. Total by Category

Calculate total ad_revenue by viewer_city.

In [ ]:
df2.groupBy('viewer_city').agg(sum('ad_revenue').alias('total_value')).orderBy(desc('total_value')).show()

## Q12. Maximum by Category

Find maximum ad_revenue in each viewer_city.

In [ ]:
df2.groupBy('viewer_city').agg(max('ad_revenue').alias('max_value')).show()

## Q13. Minimum by Category

Find minimum ad_revenue in each viewer_city.

In [ ]:
df2.groupBy('viewer_city').agg(min('ad_revenue').alias('min_value')).show()

## Q14. Having Condition

Show viewer_city groups whose total ad_revenue is greater than the overall average ad_revenue.

In [ ]:
overall = df2.select(avg('ad_revenue').alias('v')).first()['v']
df2.groupBy('viewer_city').agg(sum('ad_revenue').alias('total')).filter(col('total') > overall).show()

## Q15. Top 10 Records

Find the top 10 records by ad_revenue.

In [ ]:
df2.orderBy(desc('ad_revenue')).limit(10).show(truncate=False)

## Q16. Bottom 10 Records

Find the bottom 10 records by ad_revenue.

In [ ]:
df2.orderBy(asc('ad_revenue')).limit(10).show(truncate=False)

## Q17. Window Rank

Rank records by ad_revenue within each viewer_city.

In [ ]:
w = Window.partitionBy('viewer_city').orderBy(desc('ad_revenue'))
df2.withColumn('rank', dense_rank().over(w)).show(30, truncate=False)

## Q18. Running Total

Calculate a running total of ad_revenue within each viewer_city.

In [ ]:
w = Window.partitionBy('viewer_city').orderBy('view_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)
df2.withColumn('running_total', sum('ad_revenue').over(w)).show(30, truncate=False)

## Q19. Previous Value

Use lag to compare each record's ad_revenue with the previous record in its viewer_city.

In [ ]:
w = Window.partitionBy('viewer_city').orderBy('view_date')
df2.withColumn('previous_value', lag('ad_revenue').over(w)).show(30, truncate=False)

## Q20. Percentage Contribution

Calculate each viewer_city's contribution to total ad_revenue.

In [ ]:
tot = df2.select(sum('ad_revenue').alias('t')).first()['t']
df2.groupBy('viewer_city').agg(sum('ad_revenue').alias('total')).withColumn('pct', col('total')/lit(tot)*100).orderBy(desc('pct')).show()

## Q21. Date Extraction

Extract year and month from view_date.

In [ ]:
df2.withColumn('year', year('view_date')).withColumn('month', month('view_date')).show(20, truncate=False)

## Q22. Monthly KPI

Calculate total and average ad_revenue by month.

In [ ]:
df2.withColumn('month', date_format('view_date','yyyy-MM')).groupBy('month').agg(sum('ad_revenue').alias('total'), avg('ad_revenue').alias('average')).orderBy('month').show()

## Q23. Pivot Report

Create a pivot report with viewer_city as rows and year of view_date as columns.

In [ ]:
df2.withColumn('year', year('view_date')).groupBy('viewer_city').pivot('year').agg(sum('ad_revenue')).show()

## Q24. Conditional KPI

Count records where ad_revenue is above the category average.

In [ ]:
w=Window.partitionBy('viewer_city')
df2.withColumn('cat_avg',avg('ad_revenue').over(w)).filter(col('ad_revenue')>col('cat_avg')).groupBy('viewer_city').count().show()

## Q25. Join Aggregation

Join the DataFrames and calculate total ad_revenue by content_type.

In [ ]:
df1.join(df2, df1['content_id']==df2['content_id']).groupBy(df1['content_type']).agg(sum(df2['ad_revenue']).alias('total')).orderBy(desc('total')).show()

## Q26. Join Ranking

After joining, rank content_type by total ad_revenue.

In [ ]:
r=df1.join(df2,df1['content_id']==df2['content_id']).groupBy(df1['content_type']).agg(sum(df2['ad_revenue']).alias('total'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).show()

## Q27. Business Segmentation

Create Low, Medium and High segments for ad_revenue.

In [ ]:
df2.withColumn('segment',when(col('ad_revenue')<1000,'Low').when(col('ad_revenue')<5000,'Medium').otherwise('High')).groupBy('segment').count().show()

## Q28. Data Quality Check

Count invalid negative ad_revenue values.

In [ ]:
df2.filter(col('ad_revenue')<0).count()

## Q29. Final Business Report

Create a final report by viewer_city with count, total, average, minimum, maximum and rank.

In [ ]:
r=df2.groupBy('viewer_city').agg(count('*').alias('records'),sum('ad_revenue').alias('total'),avg('ad_revenue').alias('average'),min('ad_revenue').alias('minimum'),max('ad_revenue').alias('maximum'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).orderBy('rank').show(truncate=False)

## End of Project